# Seller-Level Master Table

This notebook builds a one-row-per-seller analytical table.

Goals:
- keep seller identity and location from the seller catalog
- summarize seller-order performance from the master order table
- capture sales, delivery, review, and product-mix signals
- validate the final grain before exporting the CSV

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

PROJECT_DIR = Path(r"D:/Data_visualization_design/ecommerce-visual-analytics/data-visualization")
DATA_DIR = Path(r"D:/Data_visualization_design/ecommerce-visual-analytics/data/E-Commerce Dataset")
MASTER_ORDER_PATH = PROJECT_DIR / "master_order_table.csv"
OUTPUT_PATH = PROJECT_DIR / "seller_level_master_table.csv"

master_order_table = pd.read_csv(
    MASTER_ORDER_PATH,
    parse_dates=[
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "review_creation_date",
        "review_answer_timestamp",
    ],
)
order_items = pd.read_csv(DATA_DIR / "order_items_dataset.csv")
products = pd.read_csv(DATA_DIR / "products_dataset.csv")
category_translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")
sellers = pd.read_csv(DATA_DIR / "sellers_dataset.csv")

product_lookup = products.merge(category_translation, on="product_category_name", how="left")
product_lookup["product_category_name_en"] = product_lookup["product_category_name_english"].fillna(product_lookup["product_category_name"])

print(f"Loaded master_order_table with {len(master_order_table):,} rows")
print(f"Loaded order_items with {len(order_items):,} rows")
print(f"Loaded sellers with {len(sellers):,} rows")
display(master_order_table.head())

Loaded master_order_table with 99,441 rows
Loaded order_items with 112,650 rows
Loaded sellers with 3,095 rows


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_count,unique_sellers,unique_products,total_freight_value,total_item_value,payment_value_total,payment_installments_total,payment_type_count,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,actual_delivery_days,delivery_delay_days,late_delivery_flag,review_risk_flag
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,1.0,1.0,8.72,29.99,38.71,3.0,2.0,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11,2017-10-12 03:43:48,8.0,-8.0,False,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,1.0,1.0,22.76,118.70,141.46,1.0,1.0,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,13.0,-6.0,False,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,1.0,1.0,19.22,159.90,179.12,3.0,1.0,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18,2018-08-22 19:07:58,9.0,-18.0,False,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,1.0,1.0,27.20,45.00,72.20,1.0,1.0,359d03e676b3c069f62cadba8dd3f6e8,5.0,NaN,O produto foi exatamente o que eu esperava e estava descrito no site e chegou bem antes da data prevista.,2017-12-03,2017-12-05 19:21:58,13.0,-13.0,False,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,1.0,1.0,8.72,19.90,28.62,1.0,1.0,e50934924e227544ba8246aeb3770dd4,5.0,NaN,NaN,2018-02-17,2018-02-18 13:02:51,2.0,-10.0,False,False


In [2]:
item_enriched = order_items.merge(
    product_lookup[["product_id", "product_category_name_en"]],
    on="product_id",
    how="left",
)

seller_item_stats = item_enriched.groupby("seller_id", as_index=False).agg(
    total_items_sold=("order_item_id", "count"),
    unique_orders_from_items=("order_id", "nunique"),
    unique_products_sold=("product_id", "nunique"),
    category_diversity=("product_category_name_en", "nunique"),
    total_item_sales_value=("price", "sum"),
    total_item_freight_value=("freight_value", "sum"),
)

seller_order_level = item_enriched.groupby(["seller_id", "order_id"], as_index=False).agg(
    seller_order_item_count=("order_item_id", "count"),
    seller_order_sales_value=("price", "sum"),
    seller_order_freight_value=("freight_value", "sum"),
)

seller_order_level = seller_order_level.merge(
    master_order_table[
        [
            "order_id",
            "customer_unique_id",
            "order_purchase_timestamp",
            "customer_city",
            "customer_state",
            "review_score",
            "delivery_delay_days",
            "actual_delivery_days",
            "late_delivery_flag",
            "review_risk_flag",
        ]
    ],
    on="order_id",
    how="left",
)

seller_order_stats = seller_order_level.groupby("seller_id", as_index=False).agg(
    orders_handled=("order_id", "nunique"),
    unique_customers=("customer_unique_id", "nunique"),
    first_order_date=("order_purchase_timestamp", "min"),
    last_order_date=("order_purchase_timestamp", "max"),
    avg_review_score=("review_score", "mean"),
    review_count=("review_score", "count"),
    late_delivery_orders=("late_delivery_flag", "sum"),
    avg_delivery_delay_days=("delivery_delay_days", "mean"),
    avg_actual_delivery_days=("actual_delivery_days", "mean"),
    avg_order_items=("seller_order_item_count", "mean"),
    avg_order_sales_value=("seller_order_sales_value", "mean"),
    avg_order_freight_value=("seller_order_freight_value", "mean"),
    review_risk_orders=("review_risk_flag", "sum"),
)

seller_level_master_table = (
    sellers
    .merge(seller_item_stats, on="seller_id", how="left")
    .merge(seller_order_stats, on="seller_id", how="left")
)

seller_level_master_table["late_delivery_rate"] = (
    seller_level_master_table["late_delivery_orders"] / seller_level_master_table["orders_handled"]
)
seller_level_master_table["review_risk_rate"] = (
    seller_level_master_table["review_risk_orders"] / seller_level_master_table["orders_handled"]
)
seller_level_master_table["seller_tenure_days"] = (
    seller_level_master_table["last_order_date"] - seller_level_master_table["first_order_date"]
).dt.days
seller_level_master_table["review_risk_flag"] = seller_level_master_table["avg_review_score"] <= 2
seller_level_master_table["active_seller_flag"] = seller_level_master_table["orders_handled"].notna()

print(f"Seller-level table rows before validation: {len(seller_level_master_table):,}")
display(seller_level_master_table.head())

Seller-level table rows before validation: 3,095


,seller_id,seller_zip_code_prefix,seller_city,seller_state,total_items_sold,unique_orders_from_items,unique_products_sold,category_diversity,total_item_sales_value,total_item_freight_value,orders_handled,unique_customers,first_order_date,last_order_date,avg_review_score,review_count,late_delivery_orders,avg_delivery_delay_days,avg_actual_delivery_days,avg_order_items,avg_order_sales_value,avg_order_freight_value,review_risk_orders,late_delivery_rate,review_risk_rate,seller_tenure_days,review_risk_flag,active_seller_flag
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,3,3,3,1,218.70,27.90,3,3,2017-05-05 16:12:29,2017-08-30 11:47:52,3.00,3,1,-6.000000,12.666667,1.000,72.90000,9.30000,1,0.333333,0.333333,116,False,True
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,41,40,30,3,11703.07,1438.73,40,40,2017-03-28 09:00:00,2018-06-06 20:01:45,4.55,40,1,-15.974359,8.743590,1.025,292.57675,35.96825,2,0.025000,0.050000,435,False,True
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,1,1,1,1,158.00,16.21,1,1,2018-07-30 12:20:10,2018-07-30 12:20:10,5.00,1,0,-13.000000,4.000000,1.000,158.00000,16.21000,0,0.000000,0.000000,0,False,True
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP,1,1,1,1,79.99,15.66,1,1,2018-08-03 00:26:04,2018-08-03 00:26:04,5.00,1,0,-16.000000,5.000000,1.000,79.99000,15.66000,0,0.000000,0.000000,0,False,True
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,1,1,1,1,167.99,31.93,1,1,2017-11-14 12:04:09,2017-11-14 12:04:09,1.00,1,1,7.000000,35.000000,1.000,167.99000,31.93000,1,1.000000,1.000000,0,True,True


In [3]:
expected_rows = len(sellers)
actual_rows = len(seller_level_master_table)

print(f"Expected seller rows: {expected_rows:,}")
print(f"Actual seller-level rows: {actual_rows:,}")
print(f"Duplicate seller_id rows: {seller_level_master_table['seller_id'].duplicated().sum()}")
print("Missing values per column (top 15):")
display(seller_level_master_table.isna().sum().sort_values(ascending=False).head(15))

seller_level_master_table.to_csv(OUTPUT_PATH, index=False)
print(f"Seller-Level Master Table saved to: {OUTPUT_PATH}")
display(seller_level_master_table.head())

Expected seller rows: 3,095
Actual seller-level rows: 3,095
Duplicate seller_id rows: 0
Missing values per column (top 15):


avg_actual_delivery_days    125
avg_delivery_delay_days     125
avg_review_score              5
seller_zip_code_prefix        0
review_risk_flag              0
seller_tenure_days            0
review_risk_rate              0
late_delivery_rate            0
review_risk_orders            0
avg_order_freight_value       0
avg_order_sales_value         0
avg_order_items               0
late_delivery_orders          0
review_count                  0
seller_id                     0
dtype: int64

Seller-Level Master Table saved to: D:\Data_visualization_design\ecommerce-visual-analytics\data-visualization\seller_level_master_table.csv


,seller_id,seller_zip_code_prefix,seller_city,seller_state,total_items_sold,unique_orders_from_items,unique_products_sold,category_diversity,total_item_sales_value,total_item_freight_value,orders_handled,unique_customers,first_order_date,last_order_date,avg_review_score,review_count,late_delivery_orders,avg_delivery_delay_days,avg_actual_delivery_days,avg_order_items,avg_order_sales_value,avg_order_freight_value,review_risk_orders,late_delivery_rate,review_risk_rate,seller_tenure_days,review_risk_flag,active_seller_flag
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,3,3,3,1,218.70,27.90,3,3,2017-05-05 16:12:29,2017-08-30 11:47:52,3.00,3,1,-6.000000,12.666667,1.000,72.90000,9.30000,1,0.333333,0.333333,116,False,True
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,41,40,30,3,11703.07,1438.73,40,40,2017-03-28 09:00:00,2018-06-06 20:01:45,4.55,40,1,-15.974359,8.743590,1.025,292.57675,35.96825,2,0.025000,0.050000,435,False,True
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,1,1,1,1,158.00,16.21,1,1,2018-07-30 12:20:10,2018-07-30 12:20:10,5.00,1,0,-13.000000,4.000000,1.000,158.00000,16.21000,0,0.000000,0.000000,0,False,True
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP,1,1,1,1,79.99,15.66,1,1,2018-08-03 00:26:04,2018-08-03 00:26:04,5.00,1,0,-16.000000,5.000000,1.000,79.99000,15.66000,0,0.000000,0.000000,0,False,True
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,1,1,1,1,167.99,31.93,1,1,2017-11-14 12:04:09,2017-11-14 12:04:09,1.00,1,1,7.000000,35.000000,1.000,167.99000,31.93000,1,1.000000,1.000000,0,True,True
